Reference-edge displacement.

How far is the Overture footprint edge from the roof edge visible in NAIP?
Sample: `draw_subsample(main, "edge_distance")` (configs/subsamples.yaml), one ±1 m pin per source.
Click the visible roof edge nearest the pin. Enter without click = skip (occluded / unclear).
Measured: signed distance of the click to the footprint union (positive = outside footprint).

In [ ]:
%env DYLD_FALLBACK_LIBRARY_PATH=/opt/homebrew/lib
%matplotlib macosx

import json

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from affine import Affine
from rasterio.windows import Window
from shapely import from_wkb
from shapely.geometry import Point
from shapely.ops import nearest_points

from thesis.data import naip
from thesis.data.eligibility import context_for_scene
from thesis.eval.subsamples import draw_subsample, one_pin_per_source
from thesis.run import REPO_ROOT, _geometry_to_crs, _valid_context_footprints, load_config

out_dir = REPO_ROOT / "data" / "dataset_v2"
imagery_config = load_config("imagery")
scene_cache = REPO_ROOT / imagery_config["scene_cache_dir"]

ANNOTATOR = "phillip"
ZOOM_PX = 48  # half-width of the zoom window around the pin (48 px = 28.8 m)

env: DYLD_FALLBACK_LIBRARY_PATH=/opt/homebrew/lib


In [ ]:
selected = pd.read_parquet(out_dir / "selection" / "sources.parquet")
pins = pd.read_parquet(out_dir / "audit" / "selected_pins.parquet")
boxes = gpd.read_parquet(out_dir / "inventory" / "boxes.parquet")
box_geometries = boxes.set_index("box_id").geometry.to_dict()
footprints = _valid_context_footprints(gpd.read_parquet(REPO_ROOT / load_config("overture")["out"]))

gold_flags = pd.read_json(out_dir / "gold" / "is_building.jsonl", lines=True)
gold = selected.merge(gold_flags, on="source_id")
main = gold.loc[gold["is_building"] & (gold["assignment"] == "main")].set_index("source_id", drop=False)
len(main)

1849

In [ ]:
sample_sources = draw_subsample(main, "edge_distance")
near = pins.loc[pins["bin_id"].isin([-1, 1])]
sample = (
    one_pin_per_source(near, sample_sources, "edge_distance")
    .merge(
        sample_sources[
            [
                "source_id",
                "box_id",
                "ecoregion",
                "naip_item_id",
                "naip_source_crs",
                "scene_window",
                "scene_geometry_wkb",
            ]
        ].reset_index(drop=True),
        on="source_id",
    )
    .reset_index(drop=True)
)
len(sample), sample["box_id"].nunique(), sample["bin_id"].value_counts().to_dict()

(197, 99, {-1: 99, 1: 98})

In [ ]:
# same crop loading as gold_dataset_filtering
item_records: dict[str, dict[str, dict]] = {}


def get_item(box_id: str, naip_item_id: str) -> dict:
    if box_id not in item_records:
        box_wgs84 = _geometry_to_crs(box_geometries[box_id], "EPSG:5070", "EPSG:4326")
        item_records[box_id] = {
            str(item["id"]): item for item in naip.search_items_for_geometry(box_wgs84, imagery_config)
        }
    return item_records[box_id][str(naip_item_id)]


def raw_crop_for(row) -> np.ndarray:
    item = get_item(str(row["box_id"]), row["naip_item_id"])
    scene_window = Window(*[int(v) for v in row["scene_window"]])
    cache_key = naip.window_cache_key(str(row["naip_item_id"]), scene_window)
    tile = naip.load_cached(scene_cache, cache_key)
    if tile is None:
        tile = naip.read_item_window(
            item, scene_window, float(imagery_config["gsd_m"]), float(imagery_config["gsd_tolerance"])
        )
        naip.store_atomic(scene_cache, cache_key, tile.array, tile.metadata)
    col_off, row_off, width, height = [int(v) for v in row["crop_window"]]
    return tile.array[:, row_off : row_off + height, col_off : col_off + width]


def footprint_union_for(row):
    scene_wgs84 = _geometry_to_crs(from_wkb(row["scene_geometry_wkb"]), "EPSG:5070", "EPSG:4326")
    return context_for_scene(footprints, scene_wgs84).to_crs(str(row["naip_source_crs"])).union_all()


def edge_measures(row, x: float, y: float) -> dict:
    """Click vs the reference edge point (footprint edge nearest the pin), not vs the edge nearest the click."""
    union = footprint_union_for(row)
    pin = Point(float(row["tip_x"]), float(row["tip_y"]))
    reference = nearest_points(union.boundary, pin)[0]
    normal = np.array([reference.x - pin.x, reference.y - pin.y])
    normal /= np.linalg.norm(normal)
    shift = np.array([x - reference.x, y - reference.y])
    # positive = visible edge lies further from the pin than the footprint edge, i.e. label distance too small
    return {
        "label_error_m": float(shift @ normal),
        "dx_m": float(shift[0]),
        "dy_m": float(shift[1]),
        "shift_m": float(np.hypot(*shift)),
    }

In [ ]:
log_path = out_dir / "gold" / f"edge_marks_{ANNOTATOR}.jsonl"
done = set()
if log_path.exists():
    done = {json.loads(line)["pin_id"] for line in log_path.read_text().splitlines()}
remaining = sample.loc[~sample["pin_id"].isin(done)].reset_index(drop=True)
i = 0
len(remaining)

184

In [ ]:
# run repeatedly: click nearest visible roof edge, Enter without click = skip
row = remaining.iloc[i]
crop = raw_crop_for(row)
tip_row, tip_col = int(row["tip_row"]), int(row["tip_col"])

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(np.transpose(crop, (1, 2, 0)))
ax.plot(tip_col, tip_row, "r+", markersize=14, markeredgewidth=1)
ax.set_xlim(tip_col - ZOOM_PX, tip_col + ZOOM_PX)
ax.set_ylim(tip_row + ZOOM_PX, tip_row - ZOOM_PX)
ax.set_title(f"[{i + 1}/{len(remaining)}] click nearest visible roof edge, Enter = skip")
clicks = plt.ginput(1, timeout=0)
plt.close(fig)

record = {
    "pin_id": row["pin_id"],
    "source_id": row["source_id"],
    "bin_id": int(row["bin_id"]),
    "exact_signed_distance_m": float(row["exact_signed_distance_m"]),
    "annotator": ANNOTATOR,
}
if clicks:
    col, r = clicks[0]
    x, y = Affine(*row["crop_transform"]) * (col + 0.5, r + 0.5)
    record.update(
        {"skipped": False, "click_col": col, "click_row": r, "click_x": x, "click_y": y, **edge_measures(row, x, y)}
    )
else:
    record.update({"skipped": True, "reason": input("skip reason: ").strip()})

with log_path.open("a") as f:
    f.write(json.dumps(record) + "\n")
print(record.get("label_error_m", "skipped"))
i += 1

KeyboardInterrupt: 

In [ ]:
marks = pd.read_json(log_path, lines=True)
clicked = marks.loc[~marks["skipped"], ["pin_id", "click_x", "click_y"]].merge(sample, on="pin_id")
clicked = clicked.join(
    pd.DataFrame([edge_measures(r, r["click_x"], r["click_y"]) for _, r in clicked.iterrows()], index=clicked.index)
)
# label_error_m is relative to the pin side; outward_m has one convention: + = visible roof edge outside footprint edge
d = clicked["exact_signed_distance_m"]
clicked["outward_m"] = np.where(d > 0, -clicked["label_error_m"], clicked["label_error_m"])
offset = clicked["outward_m"].mean()
p95 = (clicked["outward_m"] - offset).abs().quantile(0.95)

# smallest trusted bin = smallest |bin| above p95 of |outward - mean offset|
bins = sorted({abs(b["id"]) for b in load_config("bins")["bins"]})
smallest_trusted_bin = next((b for b in bins if b > p95), None)
{
    "n": len(marks),
    "skipped": int(marks["skipped"].sum()),
    "mean_offset_m": offset,
    "sd_m": clicked["outward_m"].std(),
    "p95_abs_residual_m": p95,
    "smallest_trusted_bin_m": smallest_trusted_bin,
    "mean_shift_xy_m": (clicked["dx_m"].mean(), clicked["dy_m"].mean()),
}

{'n': 197,
 'skipped': 1,
 'mean_offset_m': np.float64(-0.4051660446898931),
 'sd_m': np.float64(1.2529415597953286),
 'p95_abs_residual_m': np.float64(2.3780122038428937),
 'smallest_trusted_bin_m': 3,
 'mean_shift_xy_m': (np.float64(0.23633731346881875),
  np.float64(-0.03745089274329635))}

In [ ]:
# ±1 m label flips: visible signed distance = exact + sign(exact) * label_error; flip = sign changes
clicked["visible_signed_m"] = d + np.sign(d) * clicked["label_error_m"]
clicked["flip"] = np.sign(clicked["visible_signed_m"]) != np.sign(d)

rng = np.random.default_rng(0)
per_box = clicked.groupby("box_id")["flip"].agg(["sum", "size"])
replicates = [per_box.iloc[rng.integers(len(per_box), size=len(per_box))].sum() for _ in range(5000)]
rates = [r["sum"] / r["size"] for r in replicates]
(
    clicked["flip"].mean(),
    np.quantile(rates, [0.025, 0.975]),
    clicked.groupby("bin_id")["flip"].agg(["sum", "size", "mean"]),
)

(np.float64(0.3076923076923077),
 array([0.07692308, 0.53846154]),
         sum  size  mean
 bin_id                 
 -1        2     5  0.40
  1        2     8  0.25)

In [ ]:
# by provenance of the nearest footprint (exploratory, n per group small)
from thesis.run import _provenance_category

clicked["provenance"] = clicked["nearest_polygon_provenance"].map(_provenance_category)
clicked.groupby("provenance").agg(
    n=("flip", "size"),
    boxes=("box_id", "nunique"),
    flip_rate=("flip", "mean"),
    outward_mean=("outward_m", "mean"),
    outward_median=("outward_m", "median"),
    outward_sd=("outward_m", "std"),
)

,n,boxes,flip_rate,outward_mean,outward_median,outward_sd
provenance,,,,,,
Microsoft ML Buildings,8,8,0.25,-0.462312,-0.349752,0.851616
OpenStreetMap,5,5,0.40,-0.313733,0.183304,1.850248
